# Alpha-Gamma grid heatmaps

This notebook builds heatmaps from `completed_grid_*.csv` and `bifurcations/min_re_*.csv` for each symmetry group.


In [1]:
import Pkg; Pkg.activate("../../.")

  Activating project at `~/dev/MyCloudAtlas.jl`


In [2]:
# import Pkg
# Pkg.add("CSV")
# Pkg.add("DataFrames")

In [3]:
using CSV
using DataFrames
using CairoMakie
using Statistics
using Printf

CairoMakie.activate!()


In [4]:
base_dir = joinpath(@__DIR__, "eqb_alpha_gamma_grid")
groups = ["A", "B", "C", "D", "E", "F", "G"]

# Default grid (used only if we can't infer from data).
default_Lx_vals = collect(range(5, 15; length = 25))
default_Lz_vals = collect(range(2, 10; length = 25))


25-element Vector{Float64}:
  2.0
  2.3333333333333335
  2.6666666666666665
  3.0
  3.3333333333333335
  3.6666666666666665
  4.0
  4.333333333333333
  4.666666666666667
  5.0
  5.333333333333333
  5.666666666666667
  6.0
  6.333333333333333
  6.666666666666667
  7.0
  7.333333333333333
  7.666666666666667
  8.0
  8.333333333333334
  8.666666666666666
  9.0
  9.333333333333334
  9.666666666666666
 10.0

In [5]:
function find_index(vals::Vector{Float64}, target::Float64; atol = 1e-6)
    for (i, v) in pairs(vals)
        if isapprox(v, target; atol = atol, rtol = 0.0)
            return i
        end
    end
    return nothing
end

function centers_to_edges(vals::Vector{Float64})
    n = length(vals)
    if n == 1
        delta = 1.0
        return [vals[1] - delta / 2, vals[1] + delta / 2]
    end
    mids = (vals[1:end-1] .+ vals[2:end]) ./ 2
    left = vals[1] - (mids[1] - vals[1])
    right = vals[end] + (vals[end] - mids[end])
    return vcat(left, mids, right)
end


centers_to_edges (generic function with 1 method)

In [6]:
function read_completed_grid(path::AbstractString)
    isfile(path) || return nothing
    df = CSV.read(path, DataFrame)
    return [(row.Lx, row.Lz) for row in eachrow(df)]
end

function build_completed_mat(Lx_vals::Vector{Float64}, Lz_vals::Vector{Float64}, points)
    mat = fill(0.0, length(Lz_vals), length(Lx_vals))
    for (Lx, Lz) in points
        i = find_index(Lz_vals, Lz)
        j = find_index(Lx_vals, Lx)
        (i === nothing || j === nothing) && continue
        mat[i, j] = 1.0
    end
    return mat
end

function read_min_re(path::AbstractString)
    isfile(path) || return nothing
    df = CSV.read(path, DataFrame)
    min_map = Dict{Tuple{Float64, Float64}, Float64}()
    count_map = Dict{Tuple{Float64, Float64}, Int}()
    for row in eachrow(df)
        key = (row.Lx, row.Lz)
        min_map[key] = min(get(min_map, key, Inf), row.min_Re)
        count_map[key] = get(count_map, key, 0) + 1
    end
    return min_map, count_map
end

function build_min_re_mat(Lx_vals::Vector{Float64}, Lz_vals::Vector{Float64}, min_map)
    mat = fill(NaN, length(Lz_vals), length(Lx_vals))
    for (Lx, Lz) in keys(min_map)
        i = find_index(Lz_vals, Lz)
        j = find_index(Lx_vals, Lx)
        (i === nothing || j === nothing) && continue
        mat[i, j] = min_map[(Lx, Lz)]
    end
    return mat
end

function build_count_mat(Lx_vals::Vector{Float64}, Lz_vals::Vector{Float64}, count_map)
    mat = fill(0.0, length(Lz_vals), length(Lx_vals))
    for (Lx, Lz) in keys(count_map)
        i = find_index(Lz_vals, Lz)
        j = find_index(Lx_vals, Lx)
        (i === nothing || j === nothing) && continue
        mat[i, j] = count_map[(Lx, Lz)]
    end
    return mat
end


build_count_mat (generic function with 1 method)

In [7]:
function infer_grid_vals(completed_points, min_maps, default_Lx_vals::Vector{Float64}, default_Lz_vals::Vector{Float64})
    Lx_vals = Float64[]
    Lz_vals = Float64[]

    if completed_points !== nothing
        append!(Lx_vals, (p[1] for p in completed_points))
        append!(Lz_vals, (p[2] for p in completed_points))
    end

    if min_maps !== nothing
        min_map, _ = min_maps
        for (Lx, Lz) in keys(min_map)
            push!(Lx_vals, Lx)
            push!(Lz_vals, Lz)
        end
    end

    if isempty(Lx_vals) || isempty(Lz_vals)
        return default_Lx_vals, default_Lz_vals
    end

    return sort(unique(Lx_vals)), sort(unique(Lz_vals))
end


infer_grid_vals (generic function with 1 method)

In [21]:
function save_heatmap(out_path::AbstractString, title::AbstractString, Lx_vals::Vector{Float64}, Lz_vals::Vector{Float64}, mat; label::AbstractString = "", colormap = :turbo)
    fig = Figure(size = (900, 700))
    ax = Axis(fig[1, 1]; xlabel = "Lz", ylabel = "Lx", title = title)
    Lz_edges = centers_to_edges(Lz_vals)
    Lx_edges = centers_to_edges(Lx_vals)
    hm = heatmap!(ax, Lz_edges, Lx_edges, mat; colormap = colormap)
    Colorbar(fig[1, 2], hm; label = label)
    save(out_path, fig)
    return out_path
end


save_heatmap (generic function with 1 method)

In [22]:
for g in groups
    group_dir = joinpath(base_dir, g)
    out_dir = joinpath("./heatmaps")
    mkpath(out_dir)

    completed_path = joinpath(group_dir, "completed_grid_$(g).csv")
    completed = read_completed_grid(completed_path)

    min_re_path = joinpath(group_dir, "bifurcations", "min_re_$(g).csv")
    min_re_data = read_min_re(min_re_path)

    Lx_vals, Lz_vals = infer_grid_vals(completed, min_re_data, default_Lx_vals, default_Lz_vals)

    # if completed === nothing
    #     @warn "missing completed grid" completed_path
    # else
    #     completed_mat = build_completed_mat(Lx_vals, Lz_vals, completed)
    #     out_path = joinpath(out_dir, "heatmap_completed_$(g).png")
    #     save_heatmap(out_path, "Group $(g) - completed grid", Lx_vals, Lz_vals, completed_mat; label = "completed (1=yes)", colormap = :greys)
    #     @info "saved" out_path
    # end

    if min_re_data === nothing
        @warn "missing min_re file" min_re_path
    else
        min_map, count_map = min_re_data
        min_re_mat = build_min_re_mat(Lx_vals, Lz_vals, min_map)
        count_mat = build_count_mat(Lx_vals, Lz_vals, count_map)

        out_path = joinpath(out_dir, "heatmap_min_re_$(g).png")
        save_heatmap(out_path, "Group $(g) - min Re", Lx_vals, Lz_vals, min_re_mat; label = "min Re")
        @info "saved" out_path

        out_path = joinpath(out_dir, "heatmap_branch_count_$(g).png")
        save_heatmap(out_path, "Group $(g) - branch count", Lx_vals, Lz_vals, count_mat; label = "count")
        @info "saved" out_path
    end
end


┌ Info: saved
└   out_path = "./heatmaps/heatmap_min_re_A.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_branch_count_A.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_min_re_B.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_branch_count_B.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_min_re_C.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_branch_count_C.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_min_re_D.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_branch_count_D.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_min_re_E.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_branch_count_E.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_min_re_F.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_branch_count_F.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_min_re_G.png"
┌ Info: saved
└   out_path = "./heatmaps/heatmap_branch_count_G.png"
